In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, initcap, upper # Besoin pour Q11 et +
from pyspark.sql.types import DateType, DoubleType, IntegerType, StringType # besoin pour la Q13 et les casts

# Initialisation de la session Spark
spark = SparkSession \
    .builder \
    .appName("TradeCorp ETL") \
    .getOrCreate()

print(spark.version);

# Chemin relatif vers les CSV
PATH = "../data/"

# Création des DataFrames
# inferSchema=True permet de déduire les types de chaque colonne automatiquement
df_categories = spark.read.csv(f"{PATH}categories.csv", header=True, inferSchema=True);
df_customers = spark.read.csv(f"{PATH}customers.csv", header=True, inferSchema=True);
df_employees = spark.read.csv(f"{PATH}employees.csv", header=True, inferSchema=True);
df_orders_details = spark.read.csv(f"{PATH}order_details.csv", header=True, inferSchema=True);
df_orders = spark.read.csv(f"{PATH}orders.csv", header=True, inferSchema=True);
df_products = spark.read.csv(f"{PATH}products.csv", header=True, inferSchema=True);
df_shippers = spark.read.csv(f"{PATH}shippers.csv", header=True, inferSchema=True);
df_suppliers = spark.read.csv(f"{PATH}suppliers.csv", header=True, inferSchema=True);

# Dictionnaire de tout les DataFrames
df_collection = {"Categories" : df_categories, 
                 "Clients" : df_customers, 
                 "Employees" : df_employees, 
                 "Details des commandes" : df_orders_details, 
                 "Commandes" : df_orders, 
                 "Produits" : df_products, 
                 "Transporteurs" : df_shippers, 
                 "Fournisseurs": df_suppliers}

4.2.0


# Q11 — Valeurs nulles
Pour chaque DataFrame, compter le nombre de valeurs nulles par colonne. Utiliser une boucle et
df.filter(col(c).isNull()).count().

In [2]:
for name, df in df_collection.items():
    print(f"DataFrame: {name}")
    for c in df.columns:
        # Compter les valeurs nulles dans chaque colonne
        null_count = df.filter(col(c).isNull()).count()
        # N'afficher que les colonnes avec des valeurs nulles
        if null_count > 0:
            print(f"Nombre de valeurs nulles dans la colonne {c} : {null_count}")
    print("\n")

DataFrame: Categories
Nombre de valeurs nulles dans la colonne picture : 8


DataFrame: Clients
Nombre de valeurs nulles dans la colonne region : 60
Nombre de valeurs nulles dans la colonne postal_code : 1
Nombre de valeurs nulles dans la colonne fax : 22


DataFrame: Employees
Nombre de valeurs nulles dans la colonne region : 4
Nombre de valeurs nulles dans la colonne photo : 9
Nombre de valeurs nulles dans la colonne reports_to : 1


DataFrame: Details des commandes


DataFrame: Commandes
Nombre de valeurs nulles dans la colonne shipped_date : 21
Nombre de valeurs nulles dans la colonne ship_region : 507
Nombre de valeurs nulles dans la colonne ship_postal_code : 19


DataFrame: Produits


DataFrame: Transporteurs


DataFrame: Fournisseurs
Nombre de valeurs nulles dans la colonne region : 20
Nombre de valeurs nulles dans la colonne fax : 16
Nombre de valeurs nulles dans la colonne homepage : 24




# Q12 — Supprimer les nulls
Dans df_orders, supprimer les lignes où shipped_date est null (commandes non livrées). Dans df_products,
remplacer les valeurs nulles de unit_price par la médiane.

In [3]:
# Suppression des lignes avec shipped_date null dans df_orders
df_orders_clean = df_orders.dropna(subset=["shipped_date"])

print("Nombre de ligne avant suppression des nulls dans shipped_date : ", df_orders.count())
print("Nombre de lignes après suppression des nulls dans shipped_date : ", df_orders_clean.count())

Nombre de ligne avant suppression des nulls dans shipped_date :  830
Nombre de lignes après suppression des nulls dans shipped_date :  809


In [4]:
# Replacement des valeurs nulles de Produits (meme si il n'y en a pas dans notre jeu de données)*

# Calcul de la médiane sur les valeurs non nulles de unit_price
# Le 0.0 est pour la précision de l'estimation (entre 0 et 1), plus c'est proche de 0, plus c'est précis
# Le [0] est pour prendre la premiere valeur de la liste retournée par approxQuantile
unit_price_median = df_products.na.drop().approxQuantile("unit_price", [0.5], 0.0)[0]

# Remplacement des valeurs nulles de unit_price par la médiane
df_products_clean = df_products.fillna({"unit_price": unit_price_median})

 # Q13 — Cast des types
Dans df_orders, caster order_date, required_date et shipped_date en type DateType. Dans df_order_details,
caster unit_price en DoubleType et quantity en IntegerType.

In [5]:
print(df_orders_clean.dtypes)
print(df_orders_details.dtypes)

[('order_id', 'int'), ('customer_id', 'string'), ('employee_id', 'int'), ('order_date', 'date'), ('required_date', 'date'), ('shipped_date', 'date'), ('ship_via', 'int'), ('freight', 'double'), ('ship_name', 'string'), ('ship_address', 'string'), ('ship_city', 'string'), ('ship_region', 'string'), ('ship_postal_code', 'string'), ('ship_country', 'string')]
[('order_id', 'int'), ('product_id', 'int'), ('unit_price', 'double'), ('quantity', 'int'), ('discount', 'double')]


In [6]:
df_orders_clean = df_orders_clean.withColumn("order_date", col("order_date").cast(DateType()));
df_orders_clean = df_orders_clean.withColumn("required_date", col("required_date").cast(DateType()));
df_orders_clean = df_orders_clean.withColumn("shipped_date", col("shipped_date").cast(DateType()));

df_orders_details = df_orders_details.withColumn("unit_price", col("unit_price").cast(DoubleType()));
df_orders_details = df_orders_details.withColumn("quantity", col("quantity").cast(IntegerType()));

In [7]:
print(df_orders_clean.dtypes)
print(df_orders_details.dtypes)

[('order_id', 'int'), ('customer_id', 'string'), ('employee_id', 'int'), ('order_date', 'date'), ('required_date', 'date'), ('shipped_date', 'date'), ('ship_via', 'int'), ('freight', 'double'), ('ship_name', 'string'), ('ship_address', 'string'), ('ship_city', 'string'), ('ship_region', 'string'), ('ship_postal_code', 'string'), ('ship_country', 'string')]
[('order_id', 'int'), ('product_id', 'int'), ('unit_price', 'double'), ('quantity', 'int'), ('discount', 'double')]


# Q14 — Nettoyage des chaînes
Dans df_customers, appliquer TRIM sur toutes les colonnes texte. Mettre contact_name en title case avec
initcap(). Mettre country en majuscules avec upper().

In [8]:
df_customers.dtypes

[('customer_id', 'string'),
 ('company_name', 'string'),
 ('contact_name', 'string'),
 ('contact_title', 'string'),
 ('address', 'string'),
 ('city', 'string'),
 ('region', 'string'),
 ('postal_code', 'string'),
 ('country', 'string'),
 ('phone', 'string'),
 ('fax', 'string')]

In [9]:
# Pour chaque colonne dont le type est String, on effectue un TRIM sur la colonne
for c in df_customers.columns:
    if df_customers.schema[c].dataType == StringType():
        df_customers = df_customers.withColumn(c, trim(col(c)))

# Transformations spécifiques
df_customers = df_customers.withColumn("contact_name", initcap(col("contact_name")));
df_customers = df_customers.withColumn("country", upper(col("country")));

df_customers.head(5)


[Row(customer_id='ALFKI', company_name='Alfreds Futterkiste', contact_name='Maria Anders', contact_title='Sales Representative', address='Obere Str. 57', city='Berlin', region=None, postal_code='12209', country='GERMANY', phone='030-0074321', fax='030-0076545'),
 Row(customer_id='ANATR', company_name='Ana Trujillo Emparedados y helados', contact_name='Ana Trujillo', contact_title='Owner', address='Avda. de la Constitución 2222', city='México D.F.', region=None, postal_code='05021', country='MEXICO', phone='(5) 555-4729', fax='(5) 555-3745'),
 Row(customer_id='ANTON', company_name='Antonio Moreno Taquería', contact_name='Antonio Moreno', contact_title='Owner', address='Mataderos  2312', city='México D.F.', region=None, postal_code='05023', country='MEXICO', phone='(5) 555-3932', fax=None),
 Row(customer_id='AROUT', company_name='Around the Horn', contact_name='Thomas Hardy', contact_title='Sales Representative', address='120 Hanover Sq.', city='London', region=None, postal_code='WA1 1DP

# Q15 — Renommer les colonnes
Dans df_order_details, renommer unit_price en prix_unitaire et quantity en quantite. Dans df_orders, renommer
ship_via en shipper_id.

In [10]:
col_renamed_orders_details = {"unit_price": "prix_unitaire", "quantity" : "quantite"}

df_orders_details = df_orders_details.withColumnsRenamed(col_renamed_orders_details);

df_orders_clean = df_orders_clean.withColumnRenamed("ship_via", "shipper_id");

In [11]:
print(df_orders_details.columns)
print(df_orders_clean.columns)

['order_id', 'product_id', 'prix_unitaire', 'quantite', 'discount']
['order_id', 'customer_id', 'employee_id', 'order_date', 'required_date', 'shipped_date', 'shipper_id', 'freight', 'ship_name', 'ship_address', 'ship_city', 'ship_region', 'ship_postal_code', 'ship_country']


# Q16 — Colonnes calculées
Dans df_order_details, ajouter une colonne sous_total = prix_unitaire * quantite * (1 - discount). Arrondir à 2
décimales avec round().